<div align="left">

# 章节实践：使用 OpenCode 开发行 Softmax 算子

本章前两节介绍了 PyPTO Agent 的架构与使用方法。本节通过行 Softmax 算子完成一次完整实践：先独立编写任务定义，再用 OpenCode 驱动 Agent 生成算子，最后依据真实 Ascend NPU 上的测试结果完成独立验收。

</div>

<div align="left">

## 实践目标与完成条件

第三章里，我们已经用 `[8,8]` FP32 手工实现过行 Softmax。行 Softmax 就是按行做 Softmax；而「数值稳定的 Softmax」，指的是先减去每行的最大值再做指数和归一化，避免指数部分太大导致溢出、结果丢失精度。这一节不重复公式推导和手工写 kernel，而是把同一个算子交给 PyPTO Agent——你的任务从「把代码写对」变成「把任务定义清楚、识别关键产物、完成独立验收」。

> **说明**：本实践使用较小的固定 shape `[8,8]` 是为了让初学者专注于理解 Agent 的工作流程和任务定义方法，而非算子本身的复杂度。实际生产环境中，PyPTO Agent 同样支持动态 shape 和更复杂的算子规格。

完成条件：`custom/row_softmax/`（工作目录，Agent 生成的各种文件都会放在这里）中产生规格、独立 golden、PyPTO 实现和测试；测试在真实 Ascend NPU 上退出码为 0，且输出在 `atol=1e-3`、`rtol=1e-3` 下 `all_close=true`。本实践 `max_stage=6`（max_stage 表示 Agent 的工作最远推进到第 6 阶段，即整体精度验收，不再进入性能优化），Stage 7 性能调优不在评价范围。

</div>

<div align="left">

## 第一步：编写任务定义

参照上一节的「任务定义的五要素」和 ReLU 示例，请先不看答案，独立编写一份行 Softmax 任务定义，至少包含以下内容：

1. 最后一维数值稳定 Softmax 的数学定义（先减行最大值再归一化）；
2. 输入输出名称、`[8,8]` shape 和 FP32 dtype；
3. 独立 PyTorch golden、真实 Ascend NPU、普通随机和含较大正负值两类测试输入；
4. `atol=1e-3`、`rtol=1e-3` 与逐输出比较；
5. `max_stage=6`，精度通过后停止，不执行性能调优。

请在下方代码框中编写你的任务定义（使用 Markdown 或纯文本格式）：

</div>

<div align="left">

### 写之前：任务定义的每一段该写什么

任务定义的骨架在上一节的 ReLU 示例里已经见过：算子规格、参考实现与测试、工作流范围、禁止行为四大段。这次算子是「行 Softmax」，骨架不变，但每一段里的具体内容要跟着算子变。下表给出各段的填写要点，以及和 ReLU 对比需要改的地方：

<table style="margin-left:0;border-collapse:collapse"><thead><tr><th style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">任务分段</th><th style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">写什么</th><th style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">与 ReLU 示例不同的地方（需要改）</th></tr></thead><tbody><tr><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">算子规格</td><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">名称、输入输出、数学定义、支持范围</td><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">名称改成 <code>row_softmax</code>；shape 从 <code>[8,128]</code> 改成 <code>[8,8]</code>；数学定义改为按行做数值稳定归一化</td></tr><tr><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">参考实现与测试</td><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">golden 要求、测试输入、设备、容差</td><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">测试输入除普通随机外，要专门写明包含较大正负值的输入，用来考验 softmax 的数值稳定性</td></tr><tr><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">工作流范围</td><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">max_stage、性能目标、产物目录</td><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">产物目录从 <code>custom/relu/</code> 改成 <code>custom/row_softmax/</code>；<code>max_stage=6</code>、不做性能调优与 ReLU 相同</td></tr><tr><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">禁止行为</td><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">列出禁止做的事</td><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">实现文件名改成 <code>row_softmax_impl.py</code>，其余禁止项照抄</td></tr></tbody></table>

**黄金法则**：任务里写到的每一项，都应能在后面的产物（`SPEC.md`、golden、测试脚本）里找到对应依据。填写时对着这句话检查，不容易漏写，也不会写多余。

</div>

In [ ]:
# 请在此处编写你的行 Softmax 任务定义
# 提示：参照 ReLU 示例的结构，包含以下部分：
# - 算子规格（名称、输入输出、数学定义、支持范围）
# - 参考实现与测试（golden 要求、测试输入、设备、容差）
# - 工作流范围（max_stage、性能目标、产物目录）
# - 禁止行为

task_definition = """
# 行 Softmax 算子开发任务

## 算子规格

- 算子名称：row_softmax
- 输入：x，shape 为 [8, 8]，dtype 为 FP32
- 输出：y，shape 为 [8, 8]，dtype 为 FP32
- 数学定义：（请在此处填写数值稳定 Softmax 的公式）
- 支持范围：（请在此处说明 shape 和 dtype 要求）

## 参考实现与测试

- golden 要求：（请在此处描述 golden 的要求）
- 测试输入：（请在此处描述测试输入的类型）
- 执行设备：（请在此处说明测试设备要求）
- 精度要求：（请在此处填写容差配置）

## 工作流范围

- max_stage：（请在此处填写）
- 性能目标：（请在此处说明是否进行性能调优）
- 产物目录：（请在此处填写工作目录）

## 禁止行为

- （请在此处列出禁止的行为）
"""

print(task_definition)

<div align="left">

写完后再运行本章末尾的答案单元，对照参考任务。参考答案固定任务语义与证据条件，不限定逐字写法。

</div>

<div align="left">

## 第二步：运行 Agent 并观察执行过程

1. 把写好的任务文本粘贴到上一节部署好的 Agent 工程目录里，用上一节介绍的交互式 OpenCode 提交；
2. 运行过程中和结束后，观察 `custom/row_softmax/` 下各个阶段产物出现的先后顺序，对照上一节的阶段表，确认每个产物来自哪个智能体；
3. 打开 `MEMORY.md` 和 `.orchestrator_state.json`，确认进度已经推进到 Stage 6，且记录完整。

如果运行时间很长甚至看起来超时，先去看日志和状态文件，超时并不等于精度失败。

</div>

<div align="left">

### 运行过程中会看到什么

Agent 运行通常需要几分钟，中间产物会按阶段逐个出现。结束时 `custom/row_softmax/` 目录大致长这样：

```text
custom/row_softmax/
├── SPEC.md                # Stage 1：算子规格说明
├── API_REPORT.md          # Stage 1：接口报告
├── row_softmax_golden.py  # Stage 2：PyTorch 参考实现（标准答案）
├── DESIGN.md              # Stage 3：设计方案
├── module_interfaces.yaml # Stage 4：模块接口约定
├── row_softmax_impl.py    # Stage 5：实现代码
├── test_row_softmax.py    # Stage 5：测试脚本
├── MEMORY.md              # 多位智能体共享的叙事记忆
└── .orchestrator_state.json  # 状态机文件
```

观察要点：

1. **产物出现的顺序**与阶段表一致。哪个文件出现得晚，说明对应阶段仍在推进，不必焦虑；
2. **产物内容与任务文本互相印证**。例如打开 `SPEC.md`，应该能看到 `[8,8]`、FP32、`atol=1e-3` 等你在任务里写过的信息；
3. **状态机文件是「进度表」**。打开 `.orchestrator_state.json`，`current_stage` 应推进到 6；若 `module_status` 中出现 `failed`，说明该模块校验失败并被拦下，Agent 正在修复或重试。

> **如果某文件迟迟不出现**：先看 `.orchestrator_state.json` 停在哪个阶段、配合运行日志判断卡点，而不是盲目重跑任务。

</div>

<div align="left">

## 第三步：独立验收

按上一节的方式独立重跑生成的 `test_row_softmax.py`，然后逐项核对下面的清单。每项只能回答「满足」或「不满足」，不使用「基本正确」这类模糊表述：

<table style="margin-left:0;border-collapse:collapse"><thead><tr><th style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">#</th><th style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">验收项</th><th style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">判断依据</th></tr></thead><tbody><tr><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">1</td><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">数学定义明确为最后一维的数值稳定 Softmax</td><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">任务文本中包含先减行最大值再归一化的公式或文字描述</td></tr><tr><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">2</td><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">输入输出均明确为 <code>[8,8]</code> FP32</td><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">任务文本与 <code>SPEC.md</code>、实现签名一致</td></tr><tr><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">3</td><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">明确真实 Ascend NPU、独立 golden、两类测试输入</td><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">任务文本中有明确要求，测试脚本实际运行在 NPU 上</td></tr><tr><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">4</td><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">明确 <code>atol=1e-3</code>、<code>rtol=1e-3</code> 且逐输出比较</td><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">任务文本与测试脚本的容差配置一致</td></tr><tr><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">5</td><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">端到端测试退出码为 0，且输出 <code>all_close=true</code></td><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">由学习者本次独立运行 <code>test_row_softmax.py</code> 的实际输出确定</td></tr></tbody></table>

前四项可由任务文本与产物直接核对；第五项只能由本次真实 NPU 运行的结果确定。五项全部满足，本章实践通过。

</div>

<div align="left">

## 实践参考答案

运行下面的单元查看行 Softmax 参考任务定义和验收清单的判断依据。

</div>

In [ ]:
!cat ./answer/05.03_answer.md

<div align="left">

## 本章小结：一张知识框架图

本章没有改变 PyPTO 算子的正确性标准，而是改变了学习者的角色：从「亲手实现」变为「任务定义者 + 结果验收者」。把本章知识点收成一张图：

### 一、实践流程四步走

<table style="margin-left:0;border-collapse:collapse"><thead><tr><th style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">步骤</th><th style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">做什么</th><th style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">关键操作</th></tr></thead><tbody><tr><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">1. 编写任务定义</td><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">用五要素描述完整需求</td><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">公式、shape、dtype、容差、`max_stage` 一项不缺</td></tr><tr><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">2. 启动 Agent</td><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">交互式 OpenCode 粘贴任务</td><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">在部署好的工程目录内运行 `opencode`</td></tr><tr><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">3. 观察执行过程</td><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">对照阶段表看产物出现顺序</td><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">状态文件 `current_stage` 推进到 6</td></tr><tr><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">4. 独立验收</td><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">自己重跑测试，不看总结</td><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">`python custom/<op>/test_<op>.py`</td></tr></tbody></table>

### 二、验收的三条标准（缺一不可）

1. 测试**真实运行在 Ascend NPU 上**（不是 CPU、仿真或「实现和自身比较」）；
2. 每个输出均 `all_close=true`（`atol=1e-3`、`rtol=1e-3`），逐值在容差内；
3. 命令行**退出码为 0**，无报错退出。

### 三、注意事项

- **超时不等于失败**：先看运行日志与 `.orchestrator_state.json` 再下结论；
- **中断可续跑**：进度已落盘，在交互式界面输入「继续」即从断点恢复，已完成的阶段不重做；
- **凭据要留痕**：每个结论都应能在产物（golden、测试、状态文件）中找到证据链。

### 四、核心能力收获

性能优化可以在正确性冻结之后继续探索，但不影响本章实践是否通过。通过本章，你掌握了「任务定义者与结果验收者」这个新的使用姿势——把问题描述清楚、把质量判断标准定清楚，剩下的交给工程机制去保障。这比「写代码」本身更贴近真实的 Agent 协作场景。

</div>